# Fast-fail analysis — AE trivial inversion (latent recovery)

**Question**: can the foundation model's latent space be navigated by gradient
optimization well enough to represent true velocity models? (the
reparameterized-FWI premise)

**Data**: `notebooks/data/` — extracted from MLflow experiment
`encdec.highres.inversion` by `notebooks/data/extract_from_mlflow.py`.
Provenance and column meanings: `notebooks/data/DATA_DICTIONARY.md`.

**Campaign**: 4 refit ML models (encdec 3×3 seed 7, encdec 5×5 seed 79,
encdec 7×7 seed 571, transencdec 5×5 seed 571) × 4 latent inits ×
5 lrs × 4 epoch budgets; every run inverts the SAME 3 target models —
the hardcoded inverted test indices [1, 10, 20], hand-picked difficult
examples from the refit's test set (whose 100-model MCCV holdout is
logged as context only). The extract carries the **323 ACTIVE runs** of
the experiment (standing rule: active only); **320 form the clean set**
(the complete factorial).

Reading order: audit the cleaning (cell 2) → sweep map (3) → final-state
tables (4) → per-example results (5) → example identity (6) →
best-so-far (8) → difficulty spread (9) →
best-so-far (8) → difficulty spread (9) →
verdict (10) — then the fast-fail verdict pack (cells 11–17).

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("data") if Path("data/runs.csv").exists() else Path("notebooks/data")
runs = pd.read_csv(DATA / "runs.csv", parse_dates=["start_time_iso"])
print("preview of runs:")
display(runs.head())
print("total runs:", len(runs), "(active only — standing run-lifecycle rule)")
display(runs["group"].value_counts().rename("runs").to_frame())

sweep = runs[runs["group"] == "sweep_full_samples"].copy()
kept = sweep[sweep["kept_in_clean_set"] == True].copy()     # noqa: E712
print(f"sweep {len(sweep)} = kept {len(kept)} (one run per combo)")
assert len(runs) == 323 and len(kept) == 320, "unexpected extract shape"
assert len(kept) == kept["combo"].nunique(), "kept must be one row per combo"
# Standing rule (user, 2026-11-24, valid forever): obsolete/play runs are
# deleted from MLflow before results are taken; kept test runs are renamed.
# The extract carries ACTIVE runs only — the 137 soft-deleted runs of this
# campaign are documented in DATA_DICTIONARY.md but are not in this data.

print("\nkept runs per refit ML model:")
display(kept.groupby(["arch_family", "kernel", "seed", "refit_short"])
        .size().rename("kept").to_frame())

In [ ]:
# Sweep map: is the 4x5x4 factorial complete for every refit model?
INITS = ["trained", "zeros", "random", "randomn"]
LRS = [0.001, 0.01, 0.1, 0.5, 1.0]
BUDGETS = [50, 100, 500, 1000]
key_cols = ["refit_short", "latent_init", "lr", "max_epochs"]

present = set(map(tuple, kept[key_cols].itertuples(index=False, name=None)))
expected = {(r, i, float(lr), float(ep))
            for r in kept["refit_short"].unique()
            for i in INITS for lr in LRS for ep in BUDGETS}
print(f"grid cells present: {len(present & expected)} / {len(expected)}")
print("missing:", sorted(expected - present) or "none")

outside = kept[~kept[key_cols].apply(lambda r: tuple(r) in expected, axis=1)]
print("outside the 4x5x4 grid:", len(outside),
      "(none expected — the 200-ep probe was soft-deleted in the curation)")
assert len(outside) == 0, "unexpected non-grid runs in the active set"

# Duplicate re-runs are also absent from this extract by the standing rule:
# they were soft-deleted in the same manual curation. They were numerical
# replicates (same latent init; measured spread 0 except at the lr=1.0
# batch switch) — see DATA_DICTIONARY.md and docs/refactor-plan.md.

In [ ]:
# Final-state tables. CAVEAT: 'final' = state at schedule end; with
# cosine_annealing_wr that can be a peak-lr point (quantified in the next cells).
mse = "inversion_mse_mean_test_set"
ssim = "inversion_ssim_mean_test_set"

for ep in [1000, 500, 100, 50]:
    sub = kept[kept["max_epochs"] == ep]
    t = sub.pivot_table(index="latent_init", columns="lr", values=mse, aggfunc="mean")
    print(f"\n=== final MSE @ {ep} epochs — mean over the 4 refit ML models "
          f"(each run = mean over its 3 inverted targets, the difficult "
          f"examples [1, 10, 20]) ===")
    display(t.round(4))

print("\n=== per-init best across all 20 (lr, budget) combos ===")
per_combo = kept.groupby(["latent_init", "lr", "max_epochs"])[mse].mean().reset_index()
display(per_combo.loc[per_combo.groupby("latent_init")[mse].idxmin()].round(4))

print("\n=== per refit ML model, clean grid (all budgets) ===")
display(kept.groupby(["arch_family", "kernel", "seed"])[mse]
        .agg(["count", "mean", "min", "max"]).round(4))

t = kept[kept["max_epochs"] == 1000].pivot_table(index="latent_init", columns="lr",
                                                 values=ssim, aggfunc="mean")
print("\n=== final SSIM @ 1000 epochs ===")
display(t.round(3))

In [ ]:
# Per-example results: every run inverts the SAME 3 target examples (test-set positions
# [1, 10, 20]). The *_values MLflow metrics carry the per-example scalars
# (step t = t-th smallest inverted index — see log_evaluation_results);
# (vocabulary: 'model' = ML model; dataset items are 'examples' — legacy
#  identifiers keep the old wording: model_idx = example index)
# the extract maps them to model_idx in inversion_per_model.csv.
per_model = pd.read_csv(DATA / "inversion_per_model.csv")
pm = per_model[per_model["run_id"].isin(kept["run_id"])].copy()  # analysis set = kept (grid-only after cell 3)
print("per-example rows:", len(pm), "| runs:", pm["run_id"].nunique())

# (a) which target is hardest? per-example metrics averaged over all clean runs
print("=== per-example averages over all clean runs (difficulty of each target example) ===")
display(pm.groupby("model_idx")[["mse", "mae", "ssim"]].mean().round(4)
        .rename_axis(index="example_idx"))  # displayed as example_idx

# (b) per-model final MSE @ 1000 epochs: init x lr, one table per target
for idx, g in pm.groupby("model_idx"):
    sub = g[g["max_epochs"] == 1000]
    t = sub.pivot_table(index="latent_init", columns="lr", values="mse", aggfunc="mean")
    print(f"\n=== example (test-set position) {idx}: final MSE @ 1000 epochs ===")
    display(t.round(4))

# (c) the 3-example aggregate vs the per-example spread it hides
agg = kept.set_index("run_id")["inversion_mse_mean_test_set"].rename("agg_mean")
spread = pm.groupby("run_id")["mse"].agg(pm_min="min", pm_max="max")
c3 = pd.concat([agg, spread], axis=1).dropna()
c3["max/min"] = c3["pm_max"] / c3["pm_min"]
print("=== 3-example aggregate vs the per-example spread it hides ===")
display(c3.describe().round(3))

## Example identity — what distinguishes the three inverted examples

**Mechanism**: the per-example scalars ride in the `inversion_*_values`
metric series, logged with `step` = position in the sorted inverted list
([1, 10, 20]) — the extract maps step → example position. Positions are
0-based indices into each refit ML model's own 100-example holdout, so
the SAME position is a DIFFERENT dataset example per seed (the two
seed-571 ML models share their split).

In [ ]:
inv_ex = pd.read_csv(DATA / "inverted_examples.csv")
display(inv_ex)  # holdout position -> dataset example index, per refit ML model

# difficulty per position, per refit ML model — comment (i) made concrete
diff = pm.groupby(["refit_short", "model_idx"])["mse"].mean().round(4).unstack()
diff.columns = [f"pos {c}" for c in diff.columns]
print("=== average inversion MSE per position, per refit ML model ===")
display(diff)

In [ ]:
# Baseline gap: refit prediction (mean over its 100-example holdout) vs trivial
# inversion (mean over the 3 hand-picked DIFFICULT targets [1, 10, 20]).
# Difficulty mismatch inflates the gap; cell 7 (best-so-far) quantifies the
# final-state artifact on top of that.
base = pd.read_csv(DATA / "refit_baselines.csv").set_index("refit_short")

inv = kept[kept["max_epochs"] == 1000].pivot_table(
    index="refit_short", columns="latent_init", values=mse, aggfunc="mean")
cmp = pd.DataFrame({
    "refit_mse": base["mse_mean_test_set"],
    "inv@1000_mean_over_inits": inv.mean(axis=1),
    "inv@1000_best_init": inv.min(axis=1),
})
cmp["ratio_mean"] = cmp["inv@1000_mean_over_inits"] / cmp["refit_mse"]
cmp["refit_ssim"] = base["ssim_mean_test_set"]
inv_ssim = kept[kept["max_epochs"] == 1000].pivot_table(
    index="refit_short", columns="latent_init", values=ssim, aggfunc="mean")
cmp["inv_ssim@1000_mean"] = inv_ssim.mean(axis=1)

print("=== refit prediction vs trivial inversion ===")
display(cmp.round(4))

In [ ]:
# Convergence, PER EXAMPLE. y-axis = the INVERSION loss (the latent-optimization
# objective: model-space MSE between Generator(z) and the target example) —
# NOT the refit training loss. It is logged under the metric name 'train_loss'
# because the inversion reuses the skorch training machinery to train the latent.
# trajectories (one per target example, logged sequentially — steps restart
# per example); the extract segments them into train_loss_per_example.csv
# (model_idx = the target example's position in the refit ML model's holdout).
import matplotlib.pyplot as plt

hpe = pd.read_csv(DATA / "train_loss_per_example.csv")
curves = (hpe.sort_values("step")
             .groupby(["run_id", "model_idx", "step"], as_index=False)["value"].last())
curves = curves.merge(kept[["run_id", "combo", "refit_short", "arch_family",
                            "kernel", "seed", "latent_init", "lr", "max_epochs"]],
                      on="run_id")
# legend label: kernel size + architecture + seed (the refit run id says nothing)
curves["refit_label"] = (curves["arch_family"] + " " + curves["kernel"].astype(str)
                         + "\u00d7" + curves["kernel"].astype(str)
                         + ", seed " + curves["seed"].astype(str))

for INIT, LR in [("trained", 0.1), ("random", 0.1)]:   # bend these
    sel = curves[(curves["latent_init"] == INIT) & (curves["lr"] == LR)
                 & (curves["max_epochs"] == 1000)]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, midx in zip(axes, [1, 10, 20]):
        g = sel[sel["model_idx"] == midx]
        for refit, gg in g.groupby("refit_short"):
            ax.plot(gg["step"], gg["value"], lw=0.8, label=gg["refit_label"].iloc[0])
        ax.set_yscale("log")
        ax.set_title(f"target example at position {midx}")
        ax.set_xlabel("epoch step")
        ax.set_ylabel("inversion loss (model-space MSE)")
        ax.legend(fontsize=7)
    fig.suptitle(f"init={INIT}, lr={LR}, 1000 epochs — one curve per refit ML model")
    plt.tight_layout()
    plt.show()

# per-example best-so-far vs final state (exact per trajectory now)
bsf = curves.groupby(["run_id", "model_idx"])["value"].min().rename("best_so_far")
fin = per_model.set_index(["run_id", "model_idx"])["mse"].rename("final_mse")
bf = pd.concat([bsf, fin], axis=1).dropna()
bf["final/best"] = bf["final_mse"] / bf["best_so_far"]
print("=== per-example best-so-far vs final state (320 runs x 3 examples) ===")
display(bf.describe().round(4))
print("\nworst final/best cases:")
display(bf.sort_values("final/best", ascending=False).head(10).round(4))
print("\nbest-so-far floor vs the refit ML models' prediction MSE (~0.0009):",
      bf["best_so_far"].min())

In [ ]:
# Per-example difficulty spread (comment i): MSE varies a lot across test examples,
# so single-example readings must be difficulty-matched. Here the spread is over
# the 3 inverted targets ([1, 10, 20]) — a small sample, but it brackets the
# difficulty range those specific models span.
spread = kept[["combo", "latent_init", "lr", "max_epochs",
               "inversion_mse_min_test_set", mse,
               "inversion_mse_max_test_set", "inversion_mse_std_test_set"]].copy()
spread["max/min"] = spread["inversion_mse_max_test_set"] / spread["inversion_mse_min_test_set"]

print("per-run spread of per-example MSE over the 3 inverted targets ([1, 10, 20]):")
display(spread[["max/min", "inversion_mse_std_test_set"]].describe().round(3))

refit_spread = (base["mse_max_test_set"] / base["mse_min_test_set"]).rename("refit max/min")
print("\nrefit baseline spread on the same holdouts:")
display(refit_spread.round(2).to_frame())

ax = spread["max/min"].plot(kind="hist", bins=40, figsize=(7, 3),
                            title="per-run max/min of per-model MSE")
ax.set_xlabel("max/min")
plt.show()

# PROOF: is difficulty example-driven or ML-model-driven?
# y = max/min-style factor: MSE(position) / easiest example's MSE within the
# same ML model (easiest = 1.0; a curve's peak = that ML model's max/min).
# Reading: ML models that share the same examples must share the factor
# profile if difficulty belongs to the example. The two seed-571 models
# (7x7 encdec vs transencdec 5x5) inverted the SAME examples (429/63/239)
# with different architectures - the controlled comparison of this campaign.
prof = pm.groupby(["refit_short", "model_idx"], as_index=False)["mse"].mean()
labels = kept.drop_duplicates("refit_short").set_index("refit_short")
label_of = {r: f"{row.arch_family} {int(row.kernel)}\u00d7{int(row.kernel)}, seed {int(row.seed)}"
            for r, row in labels.iterrows()}
prof["factor"] = prof.groupby("refit_short")["mse"].transform(lambda s: s / s.min())
fig, ax = plt.subplots(figsize=(7, 4))
for refit, g in prof.groupby("refit_short"):
    g = g.sort_values("model_idx")
    ls = "--" if refit in ("b01fdd28", "32409900") else "-"
    ax.plot(g["model_idx"], g["factor"], marker="o", ls=ls, label=label_of[refit])
ax.set_xticks([1, 10, 20])
ax.set_xticklabels(["pos 1", "pos 10", "pos 20"])
ax.set_xlabel("target example position (in the refit ML model's holdout)")
ax.set_ylabel("difficulty factor (MSE / easiest example's MSE)")
ax.set_title("difficulty factor per target position — one curve per refit ML model\n"
             "(dashed = the seed-571 pair, which inverted the SAME three examples)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# quantify the controlled comparison: same examples, two different ML models
inv_ex = pd.read_csv(DATA / "inverted_examples.csv")
ex_of = inv_ex[inv_ex.refit_short == "b01fdd28"].set_index("example_pos")["dataset_example_idx"]
p571 = prof[prof["refit_short"].isin(["b01fdd28", "32409900"])]
tab = p571.pivot(index="model_idx", columns="refit_short", values="factor").round(2)
tab.columns = [label_of[c] for c in tab.columns]
tab.index = [f"pos {i} (example {ex_of[i]})" for i in tab.index]
print("=== controlled pair: same examples (429/63/239), two different ML models ===")
display(tab)
rho = tab.corr(method="spearman").iloc[0, 1]
print(f"Spearman rank correlation of the two difficulty profiles (n=3): {rho:.2f}")
print("caveat: example and seed are confounded for the other ML models — each")
print("seed holds its own examples, so this pair is the only controlled")
print("comparison in the campaign; the factor MAGNITUDES may still be")
print("ML-model-dependent even when the ordering is example-driven.")

## Fast-fail verdict pack

Four figures supporting the two core claims:

1. **Experiment grid** — the final-MSE landscape at each budget (init × lr).
2. **Best hyperparameters** — the 16-panel understanding view (4 inits × 4
   refit ML models; per-panel scale, values annotated). A reduced view will
   be planned after reviewing this one.
3. **Reachability** — convergence per init + final state at each init's
   best config: the manifold is reachable from ANY init.
4. **Gap vs baseline** — paired per example: the refit ML model's prediction
   MSE for THAT example vs the inversion's recovery of it (difficulty-matched).

5. **Convergence velocity** — the three inversion curves of the
   best ML model at its best config, with the 95%-convergence step
   marked per curve.

6. **Hardest example, per init** — convergence curves of the hardest
   inverted model at each latent init's best config, with the
   95% / 99% marks per curve.

The verdict text is yours to write in the section that follows.

In [ ]:
# 1. EXPERIMENT GRID — final-MSE landscape at each budget (init x lr)
# color = mean over the 3 target examples, pooled over the 4 refit ML models
# palette: magma (perceptual; black = best/low MSE — the emphasized end; light = worst) + PowerNorm(gamma=0.5) for low-range emphasis with a PER-BUDGET min/max scale
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
INITS = ["trained", "zeros", "random", "randomn"]
LRS = [0.001, 0.01, 0.1, 0.5, 1.0]
BUDGETS = [50, 100, 500, 1000]
grid = (kept.groupby(["latent_init", "lr", "max_epochs"], as_index=False)
        ["inversion_mse_mean_test_set"].mean())
fig, axes = plt.subplots(1, 4, figsize=(21, 4.5), sharey=True)
for ax, ep in zip(axes, BUDGETS):
    sub = grid[grid["max_epochs"] == ep]
    M = (sub.pivot(index="latent_init", columns="lr",
                   values="inversion_mse_mean_test_set")
         .reindex(index=INITS, columns=LRS))
    vmin, vmax = M.values.min(), M.values.max()   # per-budget scale
    ax.imshow(M.values, cmap="magma", norm=PowerNorm(gamma=0.5, vmin=vmin, vmax=vmax), aspect="auto")
    ax.set_xticks(range(len(LRS))); ax.set_xticklabels(LRS, rotation=45, fontsize=8)
    ax.set_yticks(range(len(INITS))); ax.set_yticklabels(INITS, fontsize=9)
    for i in range(len(INITS)):
        for j in range(len(LRS)):
            nrm = ((M.values[i, j] - vmin) / (vmax - vmin)) ** 0.5
            ax.text(j, i, f"{M.values[i, j]:.4f}", ha="center", va="center",
                    fontsize=7, color="white" if nrm < 0.45 else "black")
    ax.set_title(f"budget = {ep} epochs (own scale)", fontsize=10)
fig.suptitle("experiment grid — final MSE, lower is better (darker = better)", y=1.06)
plt.tight_layout()
plt.show()

In [ ]:
# 2. BEST HYPERPARAMETERS — 16-panel understanding view (4 inits x 4 refit ML models)
# each panel: budget (rows) x lr (cols); PER-PANEL color scale + annotated MSE
# palette: magma (perceptual; black = best/low MSE — the emphasized end; light = worst) + PowerNorm(gamma=0.5) for low-range emphasis, matching plot 1
REFITS = ["6d115baa", "9247757e", "b01fdd28", "32409900"]
labels = kept.drop_duplicates("refit_short").set_index("refit_short")
label_of = {r: f"{row.arch_family} {int(row.kernel)}\u00d7{int(row.kernel)}, seed {int(row.seed)}"
            for r, row in labels.iterrows()}
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for i, init in enumerate(INITS):
    for j, refit in enumerate(REFITS):
        sub = kept[(kept["latent_init"] == init) & (kept["refit_short"] == refit)]
        M = (sub.pivot(index="max_epochs", columns="lr",
                       values="inversion_mse_mean_test_set")
             .reindex(index=BUDGETS, columns=LRS))
        ax = axes[i, j]
        vmin, vmax = M.values.min(), M.values.max()   # per-panel scale
        ax.imshow(M.values, cmap="magma", norm=PowerNorm(gamma=0.5, vmin=vmin, vmax=vmax), aspect="auto")
        ax.set_xticks(range(len(LRS))); ax.set_xticklabels(LRS, rotation=45, fontsize=7)
        if j == 0:
            ax.set_yticks(range(len(BUDGETS))); ax.set_yticklabels(BUDGETS, fontsize=8)
            ax.set_ylabel(f"init={init}", fontsize=10)
        for a in range(len(BUDGETS)):
            for b in range(len(LRS)):
                nrm = ((M.values[a, b] - vmin) / (vmax - vmin)) ** 0.5
                ax.text(b, a, f"{M.values[a, b]:.4f}", ha="center", va="center",
                        fontsize=6.5, color="white" if nrm < 0.45 else "black")
        ax.set_title(label_of[refit], fontsize=9)
fig.suptitle("best hyperparameters — rows: budget, cols: lr; values = final MSE (mean over 3 examples)",
             y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# 3. REACHABILITY — best refit ML model only (encdec 5x5, seed 79: lowest
# refit prediction MSE, 0.00083). Column a: convergence per init — the BOLD
# curve is the mean of the THREE inverted examples' trajectories (faint).
# Column b: per init at its best config, a boxplot of the 3 per-example
# final MSEs (mean marker + quartiles).
BEST = "9247757e"
hpe = pd.read_csv(DATA / "train_loss_per_example.csv")
cv = hpe[(hpe["refit_short"] == BEST) & (hpe["lr"] == 0.1)
         & (hpe["max_epochs"] == 1000)]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 4.5))
for init in INITS:
    g = cv[cv["latent_init"] == init]
    for midx, gg in g.groupby("model_idx"):
        gg = gg.sort_values("step")
        axL.plot(gg["step"], gg["value"], lw=0.7, alpha=0.3, color=f"C{INITS.index(init)}")
    mean = g.groupby("step", as_index=False)["value"].mean().sort_values("step")
    axL.plot(mean["step"], mean["value"], lw=1.4, color=f"C{INITS.index(init)}",
             label=f"{init} (mean of 3 examples)")
axL.set_yscale("log")
axL.set_xlabel("epoch step")
axL.set_ylabel("inversion loss (model-space MSE)")
axL.set_title("convergence per init — best ML model\n(faint: each example; bold: their mean)")
axL.legend(fontsize=7)

sub = kept[kept["refit_short"] == BEST]
best_cfg = sub.loc[sub.groupby("latent_init")["inversion_mse_mean_test_set"].idxmin()]
best_cfg = best_cfg.set_index("latent_init").loc[INITS].reset_index()
data, xlabels = [], []
for _, row in best_cfg.iterrows():
    run = kept[(kept["refit_short"] == BEST) & (kept["latent_init"] == row["latent_init"])
               & (kept["lr"] == row["lr"]) & (kept["max_epochs"] == row["max_epochs"])]
    ex = per_model[per_model["run_id"].isin(run["run_id"])].sort_values("model_idx")
    data.append(list(ex["mse"]))
    xlabels.append(f"{row['latent_init']}\nlr={row['lr']}, {int(row['max_epochs'])}ep")
axR.boxplot(data, showmeans=True,
            meanprops={"marker": "D", "markerfacecolor": "C1", "markeredgecolor": "C1"})
axR.set_xticklabels(xlabels, fontsize=8)
axR.set_ylabel("final MSE (3 target examples)")
axR.set_title("final state at each init's best config\n(box: quartiles; diamond: mean)")
plt.tight_layout()
plt.show()

In [ ]:
# 4. GAP vs BASELINE — two-column slopegraph: AE models (their prediction)
# on the LEFT, the inverted velocity models (latent recovery) on the RIGHT.
# Best refit ML model only (encdec 5x5, seed 79); for each latent init only
# its best config; each target example keeps ONE color across all lines.
BEST = "9247757e"
rpe = pd.read_csv(DATA / "refit_per_example.csv")
inv_ex = pd.read_csv(DATA / "inverted_examples.csv")
ex_name = {int(r.example_pos): f"pos {int(r.example_pos)} (example {int(r.dataset_example_idx)})"
           for r in inv_ex[inv_ex.refit_short == BEST].itertuples()}
colors = {1: "C0", 10: "C1", 20: "C2"}
base = rpe[rpe["refit_short"] == BEST].set_index("step")
inv = per_model[per_model["refit_short"] == BEST].sort_values("model_idx")
sub = kept[kept["refit_short"] == BEST]
best_cfg = sub.loc[sub.groupby("latent_init")["inversion_mse_mean_test_set"].idxmin()]
best_cfg = best_cfg.set_index("latent_init").loc[INITS].reset_index()
fig, ax = plt.subplots(figsize=(8, 5.5))
ratios_by_ex = {1: [], 10: [], 20: []}
cfg_labels = []
for mi in (1, 10, 20):
    b = float(base.loc[mi, "mse"])   # the AE model's prediction of this example
    ax.scatter([0], [b], color=colors[mi], s=60, zorder=3)
for _, row in best_cfg.iterrows():
    run = kept[(kept["refit_short"] == BEST) & (kept["latent_init"] == row["latent_init"])
               & (kept["lr"] == row["lr"]) & (kept["max_epochs"] == row["max_epochs"])]
    ex = inv[inv["run_id"].isin(run["run_id"])].sort_values("model_idx")
    cfg_labels.append(f"lr={row['lr']}, {int(row['max_epochs'])}ep")
    for _, erow in ex.iterrows():
        mi = int(erow["model_idx"])
        b = float(base.loc[mi, "mse"])
        v = float(erow["mse"])
        ratios_by_ex[mi].append(v / b)
        ax.plot([0, 1], [b, v], "-", color=colors[mi], alpha=0.5, lw=1.1)
        ax.scatter([1], [v], color=colors[mi], s=60, zorder=3)
        ax.annotate(f"{v/b:.1f}x", (1.04, v), fontsize=7.5, color=colors[mi])
ax.set_yscale("log")
ax.set_xlim(-0.3, 1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["AE models\n(prediction)", "inverted velocity models\n(latent recovery)"],
                   fontsize=10)
ax.set_ylabel("model-space MSE (log)")
ax.set_title("gap vs baseline — best refit ML model (encdec 5x5, seed 79)\n"
             "each target example keeps one color across the 4 latent inits' best configs\n"
             f"(right-column configs: {'; '.join(cfg_labels)})")
ax.legend(handles=[plt.Line2D([], [], color=colors[mi], marker="o", label=ex_name[mi])
                   for mi in (1, 10, 20)], fontsize=8)
plt.tight_layout()
plt.show()
import statistics
all_r = [r for rs in ratios_by_ex.values() for r in rs]
print(f"median ratio inversion/prediction over the 12 lines: {statistics.median(all_r):.1f}x")
for mi in (1, 10, 20):
    print(f"  {ex_name[mi]}: median {statistics.median(ratios_by_ex[mi]):.1f}x "
          f"across the 4 inits' best configs")

In [ ]:
# 5. CONVERGENCE VELOCITY — the best ML model (encdec 5x5, seed 79) at its
# best config (lowest final MSE): the three inversion curves, one per
# target example. Vertical lines per curve: the FIRST step whose loss is
# within 5% (dashed) / 1% (dotted) of that curve's final value
# (95% / 99% converged).
BEST = "9247757e"
hpe = pd.read_csv(DATA / "train_loss_per_example.csv")
inv_ex = pd.read_csv(DATA / "inverted_examples.csv")
ex_name = {int(r.example_pos): int(r.dataset_example_idx)
           for r in inv_ex[inv_ex.refit_short == BEST].itertuples()}
colors = {1: "C0", 10: "C1", 20: "C2"}
sub = kept[kept["refit_short"] == BEST]
best_run = sub.loc[sub["inversion_mse_mean_test_set"].idxmin()]
print(f"best run: init={best_run['latent_init']}, lr={best_run['lr']}, "
      f"{int(best_run['max_epochs'])} epochs "
      f"(final MSE {best_run['inversion_mse_mean_test_set']:.4f})")
curves = hpe[hpe["run_id"] == best_run["run_id"]]
fig, ax = plt.subplots(figsize=(9, 5))
for mi in (1, 10, 20):
    g = (curves[curves["model_idx"] == mi]
         .groupby("step", as_index=False)["value"].last().sort_values("step"))
    final = g["value"].iloc[-1]
    last_step = int(g["step"].iloc[-1])
    step95 = int(g.loc[g["value"] <= 1.05 * final, "step"].iloc[0])
    step99 = int(g.loc[g["value"] <= 1.01 * final, "step"].iloc[0])
    ax.plot(g["step"], g["value"], color=colors[mi], lw=1.3,
            label=f"example {ex_name[mi]} (pos {mi})")
    ax.axvline(step95, color=colors[mi], ls="--", lw=1, alpha=0.6)
    ax.axvline(step99, color=colors[mi], ls=":", lw=1, alpha=0.45)
    ax.annotate(f"{step95}", (step95, 0.93), xycoords=("data", "axes fraction"),
                fontsize=7.5, color=colors[mi], rotation=90, va="top")
    ax.annotate(f"{step99}", (step99, 0.55), xycoords=("data", "axes fraction"),
                fontsize=7.5, color=colors[mi], rotation=90, va="top")
    print(f"  example {ex_name[mi]} (pos {mi}): 95% at step {step95} "
          f"({step95/last_step*100:.0f}% of the budget), 99% at step {step99} "
          f"({step99/last_step*100:.0f}%); final MSE {final:.4f}")
ax.set_yscale("log")
ax.set_xlabel("epoch step")
ax.set_ylabel("inversion loss (model-space MSE)")
ax.set_title("convergence velocity — best ML model at its best config")
handles, labels = ax.get_legend_handles_labels()
handles += [plt.Line2D([], [], color="k", ls="--", lw=1,
                       label="95% converged (within 5% of final)"),
            plt.Line2D([], [], color="k", ls=":", lw=1,
                       label="99% converged (within 1% of final)")]
ax.legend(handles=handles, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# 6. CONVERGENCE VELOCITY — HARDEST EXAMPLE — best ML model (encdec 5x5,
# seed 79); the single hardest inverted model (highest mean final MSE
# across the inits' best configs); 4 curves = the 4 latent inits, each at
# its best config (same configs as the slopegraph). Dashed/dotted vertical
# lines: first step within 5% / 1% of that curve's final value
# (95% / 99% converged). Colors follow the reachability plot (per init).
BEST = "9247757e"
hpe = pd.read_csv(DATA / "train_loss_per_example.csv")
sub = kept[kept["refit_short"] == BEST]
pm_best = per_model[per_model["refit_short"] == BEST]
hard_idx = int(pm_best.groupby("model_idx")["mse"].mean().idxmax())
hard_mean = float(pm_best.groupby("model_idx")["mse"].mean().max())
print(f"hardest inverted model (best ML model): pos {hard_idx} "
      f"(example {ex_name[hard_idx]}), mean final MSE {hard_mean:.4f} "
      f"across the inits' best configs")
best_cfg = sub.loc[sub.groupby("latent_init")["inversion_mse_mean_test_set"].idxmin()]
best_cfg = best_cfg.set_index("latent_init").loc[INITS].reset_index()
fig, ax = plt.subplots(figsize=(9.5, 5))
for _, row in best_cfg.iterrows():
    init = row["latent_init"]
    run = kept[(kept["refit_short"] == BEST) & (kept["latent_init"] == init)
               & (kept["lr"] == row["lr"]) & (kept["max_epochs"] == row["max_epochs"])]
    g = (hpe[hpe["run_id"].isin(run["run_id"]) & (hpe["model_idx"] == hard_idx)]
         .groupby("step", as_index=False)["value"].last().sort_values("step"))
    final = g["value"].iloc[-1]
    last_step = int(g["step"].iloc[-1])
    step95 = int(g.loc[g["value"] <= 1.05 * final, "step"].iloc[0])
    step99 = int(g.loc[g["value"] <= 1.01 * final, "step"].iloc[0])
    color = f"C{INITS.index(init)}"
    ax.plot(g["step"], g["value"], color=color, lw=1.3,
            label=f"{init} — lr={row['lr']}, {int(row['max_epochs'])}ep")
    ax.axvline(step95, color=color, ls="--", lw=1, alpha=0.6)
    ax.axvline(step99, color=color, ls=":", lw=1, alpha=0.45)
    ax.annotate(f"{step95}", (step95, 0.95), xycoords=("data", "axes fraction"),
                fontsize=7.5, color=color, rotation=90, va="top")
    ax.annotate(f"{step99}", (step99, 0.60), xycoords=("data", "axes fraction"),
                fontsize=7.5, color=color, rotation=90, va="top")
    print(f"  {init:8s} lr={row['lr']}, {int(row['max_epochs'])}ep: "
          f"95% at step {step95} ({step95/last_step*100:.0f}%), "
          f"99% at step {step99} ({step99/last_step*100:.0f}% of {last_step}); "
          f"final MSE {final:.4f}")
ax.set_yscale("log")
ax.set_xlabel("epoch step")
ax.set_ylabel("inversion loss (model-space MSE)")
ax.set_title(f"convergence velocity — hardest inverted model (pos {hard_idx}, example {ex_name[hard_idx]})\n"
             "best ML model (encdec 5x5, seed 79), each latent init at its best config")
handles, labels = ax.get_legend_handles_labels()
handles += [plt.Line2D([], [], color="k", ls="--", lw=1,
                       label="95% converged (within 5% of final)"),
            plt.Line2D([], [], color="k", ls=":", lw=1,
                       label="99% converged (within 1% of final)")]
ax.legend(handles=handles, fontsize=8)
plt.tight_layout()
plt.show()


## Verdict — *draft after exploring*

**Findings**
- (fill after running the cells — the numbers you trust are the ones you produced)

**Caveats**
- final-state reporting at warm-restart peaks → quantified in the convergence cell; fix lands in the final experiments
- per-model difficulty variance → spread cell; single-example readings must be difficulty-matched

**Next (post-refactor AE+VAE rerun checklist)**
- best-so-far / cold-end reporting
- `init=encoder-of-target` — measures the manifold-coverage ceiling
- per-example paired analysis
- 10-seed MCCV (ADR-0002)